# 오디오 분류 (멜 스펙트로그램 + CNN) — Free Spoken Digits (PyTorch) — Colab

음성 파형을 **멜 스펙트로그램 이미지**로 바꿔 CNN으로 분류하는 오디오 기본 예제입니다.

- 핵심 흐름: `.wav` 파형 → **멜 스펙트로그램** → 2D CNN 분류(말한 숫자 0~9).
- IOAI 2025 GAITE **Synthetic Speech Detector**(합성 음성 탐지) 과제의 기반 기법입니다. (공식 해법도 스펙트로그램+ResNet 사용 — 동일 파이프라인, 라벨만 다름)
- Kaggle **데이터셋** [Free Spoken Digit Dataset](https://www.kaggle.com/datasets/joserzapata/free-spoken-digit-dataset-fsdd) 사용(가볍고 빠름). 토큰이 **노트북에 내장**되어 바로 실행됩니다.

> ⚙️ GPU 권장(작아서 CPU도 가능). ⚠️ 노트북에 개인 API 토큰이 평문으로 들어 있습니다. 외부 공유 금지.

## 0. Setup — 라이브러리 설치 & Kaggle 자격증명

In [ ]:
import sys, subprocess
for pkg in ["kaggle", "torch", "torchaudio", "scipy", "numpy", "matplotlib"]:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)
print("라이브러리 준비 완료")

In [ ]:
import os
os.environ["KAGGLE_API_TOKEN"] = "KGAT_5dd261fded8e0d7eb2c29d8d65dfabea"
print("Kaggle 자격증명 설정 완료 (내장 토큰)")

## 1. Kaggle에서 오디오 데이터셋 다운로드 (FSDD)

In [ ]:
import os, glob
WORK_DIR = "/content" if os.path.isdir("/content") else os.getcwd()
from kaggle.api.kaggle_api_extended import KaggleApi
api = KaggleApi(); api.authenticate()
api.dataset_download_files("joserzapata/free-spoken-digit-dataset-fsdd", path=WORK_DIR, unzip=True, quiet=False)
wavs = glob.glob(os.path.join(WORK_DIR, "**", "*.wav"), recursive=True)
print("wav 파일 수:", len(wavs), "| 예:", os.path.basename(wavs[0]))

## 2. 파형 → 멜 스펙트로그램

파일명 형식은 `숫자_화자_인덱스.wav` 이며 라벨은 맨 앞 숫자입니다. 8kHz 로 통일하고 고정 길이로 자른 뒤 멜 스펙트로그램(로그)으로 변환합니다.

In [ ]:
import torch, torchaudio, numpy as np
from scipy.io import wavfile

SR = 8000; N_MELS = 64; FIXED = 8000  # 1초
melspec = torchaudio.transforms.MelSpectrogram(sample_rate=SR, n_fft=512, hop_length=128, n_mels=N_MELS)

def wav_to_logmel(path):
    sr, data = wavfile.read(path)            # scipy 로 wav 로드(코덱 불필요)
    wav = torch.tensor(data, dtype=torch.float32)
    if wav.ndim > 1: wav = wav.mean(1)       # 모노
    wav = wav / (wav.abs().max() + 1e-8)     # 정규화
    wav = wav.unsqueeze(0)                    # (1, T)
    if sr != SR: wav = torchaudio.functional.resample(wav, sr, SR)
    if wav.size(1) < FIXED: wav = torch.nn.functional.pad(wav, (0, FIXED - wav.size(1)))
    else: wav = wav[:, :FIXED]
    m = melspec(wav)                          # (1, n_mels, T)
    return torch.log1p(m)

def label_of(path): return int(os.path.basename(path).split("_")[0])

sample = wav_to_logmel(wavs[0])
print("logmel shape:", tuple(sample.shape), "| label 예:", label_of(wavs[0]))

## 3. 데이터셋 구성 (전부 메모리에 로드)

In [ ]:
from torch.utils.data import TensorDataset, DataLoader

X = torch.stack([wav_to_logmel(p)[0] for p in wavs])  # (N, n_mels, T)
X = X.unsqueeze(1)                                      # (N, 1, n_mels, T)
y = torch.tensor([label_of(p) for p in wavs])
# 정규화
X = (X - X.mean()) / (X.std() + 1e-6)

n = len(X); g = torch.Generator().manual_seed(42); perm = torch.randperm(n, generator=g)
cut = int(n*0.9); tr, va = perm[:cut], perm[cut:]
train_loader = DataLoader(TensorDataset(X[tr], y[tr]), batch_size=64, shuffle=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device, "| X:", tuple(X.shape), "| train:", len(tr), "val:", len(va))

## 4. CNN 분류기 & 학습

In [ ]:
import torch.nn as nn

class AudioCNN(nn.Module):
    def __init__(self, n_classes=10):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1,16,3,padding=1), nn.BatchNorm2d(16), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16,32,3,padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32,64,3,padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.AdaptiveAvgPool2d(1),
        )
        self.fc = nn.Linear(64, n_classes)
    def forward(self, x): return self.fc(self.net(x).flatten(1))

model = AudioCNN().to(device)
opt = torch.optim.Adam(model.parameters(), lr=1e-3); crit = nn.CrossEntropyLoss()
Xva, yva = X[va].to(device), y[va]
for epoch in range(1, 16):
    model.train()
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        opt.zero_grad(); crit(model(xb), yb).backward(); opt.step()
    model.eval()
    with torch.no_grad():
        acc = (model(Xva).argmax(1).cpu() == yva).float().mean().item()
    if epoch % 3 == 0 or epoch == 1: print(f"epoch {epoch:2d} | val acc {acc:.4f}")
print(f"최종 val accuracy: {acc:.4f}")

## 5. 멜 스펙트로그램 시각화

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 5, figsize=(14, 3))
for k in range(5):
    ax[k].imshow(X[va[k]][0].numpy(), origin="lower", aspect="auto", cmap="magma")
    ax[k].set_title(f"label {int(y[va[k]])}"); ax[k].axis("off")
plt.suptitle("Log-Mel Spectrograms"); plt.show()

## 🏁 제출 안내

이 노트북은 **오디오 분류(스펙트로그램+CNN)** 연습용 데모입니다(제출 리더보드 없음).

- 데이터 출처: **[Free Spoken Digit Dataset](https://www.kaggle.com/datasets/joserzapata/free-spoken-digit-dataset-fsdd)**

> IOAI 2025 GAITE *Synthetic Speech Detector* 과제의 기반 기법(파형→멜 스펙트로그램→CNN/ResNet)입니다. 라벨을 '진짜/합성'으로 바꾸면 동일 파이프라인으로 합성음성 탐지가 됩니다.